# Phase A Sanity Check — Model Stitching for Audio (Colab version)

Self-contained notebook version of the Phase A pipeline. Runs the **self-stitch identity check** on `wav2vec2-base`: for every encoder layer, stitch it to every other layer (within the same model) and check whether each layer is most similar to itself.

**Before running:** `Runtime -> Change runtime type -> T4 GPU` (or any available GPU).

This is meant to get you a real result *today*, while waiting on Unity HPC access. Once you're on Unity, the equivalent script is `scripts/run_phase_a_sanity.py` in the main project — same logic, same checks, just running on cluster-scale compute instead.

## 1. Install dependencies

In [ ]:
!pip install -q transformers datasets soundfile

## 2. Imports and device check

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from transformers import AutoModel, AutoFeatureExtractor
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("WARNING: no GPU detected. Go to Runtime -> Change runtime type -> GPU, then re-run this cell.")

## 3. Load the model and a small audio sample

`wav2vec2-base` has 12 encoder layers and ~95M parameters — small enough to run comfortably on a free Colab GPU. We use a tiny public LibriSpeech dev sample just to prove the pipeline works; swap in your real Phase A dataset (e.g. a SUPERB task subset) once you move past this smoke test.

In [ ]:
MODEL_NAME = "facebook/wav2vec2-base"
NUM_EXAMPLES = 32   # keep this small for a fast first run; increase once it's working

model = AutoModel.from_pretrained(MODEL_NAME, output_hidden_states=True).eval().to(device)
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
num_layers = model.config.num_hidden_layers
print(f"Loaded {MODEL_NAME}: {num_layers} encoder layers, do_stable_layer_norm={model.config.do_stable_layer_norm}")

ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
ds = ds.select(range(min(NUM_EXAMPLES, len(ds))))
waveforms = [x["array"] for x in ds["audio"]]
print(f"Loaded {len(waveforms)} audio examples")

## 4. Extract hidden states at every layer (once, cached and reused for every layer pair)

In [ ]:
inputs = feature_extractor(waveforms, sampling_rate=16000, return_tensors="pt", padding=True)
input_values = inputs.input_values.to(device)

with torch.no_grad():
    out = model(input_values)
    hidden_states = out.hidden_states  # tuple of (num_layers + 1) tensors, each (B, T, C)

print(f"Got {len(hidden_states)} hidden-state tensors, each shaped {tuple(hidden_states[0].shape)}")

## 5. Stitching primitives

Minimal, self-contained versions of `StitchLayer` and the Direct-Matching closed-form init from the main project's `src/stitching.py`. Same math, copied inline so this notebook has zero dependency on the repo.

In [ ]:
class StitchLayer(nn.Module):
    """Linear per-frame affine stitcher (the only trainable component)."""
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.net = nn.Linear(in_dim, out_dim)

    def forward(self, x):
        return self.net(x)


def direct_matching_init(stitch_layer, source_feats, target_feats):
    """Closed-form affine least-squares fit (Moore-Penrose pseudoinverse),
    used to warm-start the stitch layer before LFM refinement."""
    K, T, C_in = source_feats.shape
    C_out = target_feats.shape[-1]
    A = source_feats.reshape(-1, C_in)
    A = torch.cat([A, torch.ones(A.shape[0], 1, device=A.device)], dim=1)
    B = target_feats.reshape(-1, C_out)
    theta = torch.linalg.lstsq(A, B).solution
    W, b = theta[:-1].T, theta[-1]
    with torch.no_grad():
        stitch_layer.net.weight.copy_(W)
        stitch_layer.net.bias.copy_(b)
    return stitch_layer


def lfm_loss(stitched_feat, target_feat):
    """Layer Feature Matching: label-free MSE at the stitch point."""
    return F.mse_loss(stitched_feat, target_feat)

## 6. Run the self-stitch sweep

For every (source layer, target layer) pair *within this same model*, fit a stitch layer (Direct Matching init + short LFM refinement) and record the final loss. Lower loss = higher similarity.

In [ ]:
LAYERS = list(range(1, num_layers + 1))   # hidden_states[0] is the conv-feature input, skip it
LFM_EPOCHS = 200
LFM_LR = 1e-3

scores = {}
total_pairs = len(LAYERS) * len(LAYERS)
done = 0

for i in LAYERS:
    src_feat = hidden_states[i].detach()
    for j in LAYERS:
        tgt_feat = hidden_states[j].detach()  # same T, same model -> no temporal resampling needed

        stitch = StitchLayer(src_feat.shape[-1], tgt_feat.shape[-1]).to(device)
        direct_matching_init(stitch, src_feat, tgt_feat)

        opt = torch.optim.AdamW(stitch.parameters(), lr=LFM_LR)
        for _ in range(LFM_EPOCHS):
            opt.zero_grad()
            loss = lfm_loss(stitch(src_feat), tgt_feat)
            loss.backward()
            opt.step()

        with torch.no_grad():
            final_loss = lfm_loss(stitch(src_feat), tgt_feat).item()
        scores[(LAYERS.index(i), LAYERS.index(j))] = -final_loss  # similarity = negative loss

        done += 1
        print(f"\r  [{done}/{total_pairs}] layer {i} -> layer {j}: loss={final_loss:.6f}", end="")

print("\nDone.")

## 7. Sanity check: is every layer most similar to itself?

This is the mandatory, cheap check from the reliability paper (Balogh & Jelasity 2025). If it fails, don't scale up yet — debug learning rate, epoch count, or stitch layer capacity first.

In [ ]:
n = len(LAYERS)
sim = np.zeros((n, n))
for (a, b), v in scores.items():
    sim[a, b] = v

correct = 0
failures = []
for row in range(n):
    best_col = int(np.argmax(sim[row]))
    if best_col == row:
        correct += 1
    else:
        failures.append((LAYERS[row], LAYERS[best_col], sim[row, best_col], sim[row, row]))

accuracy = correct / n
print(f"Self-Stitch Sanity Check ({MODEL_NAME})")
print(f"Accuracy: {accuracy:.2%}")
if failures:
    print("Failures (true_layer -> preferred_layer, preferred_score vs true_score):")
    for true_l, pref_l, pref_s, true_s in failures:
        print(f"  layer {true_l} -> preferred {pref_l}  ({pref_s:.4f} vs {true_s:.4f})")
else:
    print("No failures. Pipeline passes the sanity check.")

## 8. Visualize the similarity matrix

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(sim, cmap="viridis")
ax.set_xlabel("Target layer")
ax.set_ylabel("Source layer")
ax.set_xticks(range(n)); ax.set_xticklabels(LAYERS)
ax.set_yticks(range(n)); ax.set_yticklabels(LAYERS)
ax.set_title(f"Self-stitch similarity ({MODEL_NAME})\n(diagonal should be brightest in each row)")
plt.colorbar(im, ax=ax, label="similarity (-loss)")
plt.tight_layout()
plt.show()

## Next steps

- **If the sanity check passed:** you have real evidence the pipeline logic is sound. When Unity access comes through, run the equivalent `scripts/run_phase_a_sanity.py` there to scale up (more examples, more epochs, larger models) and move on to the cross-model correspondence check and the TLT/FFM comparison.
- **If it failed:** try increasing `LFM_EPOCHS`, tuning `LFM_LR`, or checking that `NUM_EXAMPLES` is large enough to fit a stable affine map. A persistent failure after tuning is itself a real, reportable finding — not necessarily a bug.
- This notebook intentionally uses a **linear** stitch layer with Direct Matching initialization only. The main project also supports MLP/conv stitch layers and the LFM/TLT/FFM training-objective comparison — see `src/stitching.py` in the full repo for those.